# ThermoNet: single-cube baseline

Select **Runtime → Change runtime type → GPU**. This notebook trains the deterministic U-Net on the single-cube dataset. It copies data to the runtime disk and saves checkpoints and metrics to Drive.

On your Mac, run the following from the ThermoFlow project folder, then upload the archive to `My Drive/ThermoFlow/`:

```bash
zip -r -0 thermoflow_single.zip train_thermonet.py thermonet.py datasets/single_cube
```

The archive should contain the two Python files and `datasets/single_cube/sample_*.npz`. Do not upload the virtual environment. The source dataset is unchanged.


In [ ]:
import torch
import numpy as np
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime, then reconnect and rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import shutil
import zipfile
import os

archive = Path("/content/drive/MyDrive/ThermoFlow/thermoflow_single.zip")
project = Path("/content/ThermoFlow")
run = Path("/content/drive/MyDrive/ThermoFlow/runs/single_cube")
if not archive.is_file():
    raise FileNotFoundError(f"Upload the project/data archive to {archive}")
local_archive = Path("/content/thermoflow_single.zip")
shutil.copyfile(archive, local_archive)
project.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(local_archive) as bundle:
    for name in bundle.namelist():
        target = (project / name).resolve()
        if project.resolve() not in target.parents:
            raise ValueError(f"Unexpected archive path: {name}")
    bundle.extractall(project)
os.chdir(project)
print("Specimens:", len(list(Path("datasets/single_cube").glob("sample_*.npz"))))
print("Checkpoints:", run)


## Train or resume

The script groups identical interiors before creating a random 70/15/15 split. Normalization and BCE class weighting use only training data. It minimizes weighted BCE plus soft Dice, selects the best checkpoint by validation loss, and evaluates the test set after training.

The command below uses clean videos. To test measurement noise, start a **new output directory** and add `--noise-std 0.05` (kelvin). Keep the other settings unchanged when resuming. Each epoch saves `last.pt`; `best.pt` holds the best validation checkpoint. A disconnection can lose the unfinished epoch, while completed checkpoints remain on Drive.


In [ ]:
import subprocess
import sys

command = [sys.executable, "-u", "train_thermonet.py",
           "--data", "datasets/single_cube", "--output", str(run),
           "--epochs", "100", "--batch-size", "16", "--device", "cuda",
           "--num-workers", "2", "--cache"]
if (run / "last.pt").is_file():
    command += ["--resume", str(run / "last.pt")]
subprocess.run(command, check=True)


In [ ]:
import json
import matplotlib.pyplot as plt

history = json.loads((run / "history.json").read_text())
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for split in ("train", "validation"):
    axes[0].plot(epochs, [r[split]["loss"] for r in history], label=split)
    axes[1].plot(epochs, [r[split]["dice"] for r in history], label=split)
axes[0].set(xlabel="Epoch", ylabel="Weighted BCE + soft Dice loss")
axes[1].set(xlabel="Epoch", ylabel="Thresholded 3D Dice", ylim=(0, 1))
for ax in axes:
    ax.legend()
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()
print(json.dumps(json.loads((run / "test_metrics.json").read_text()), indent=2))


## Saved outputs

- `manifest.json`: specimen hashes and the frozen split, with relative filenames.
- `normalization.json`: training-only temperature mean, standard deviation, and class weight.
- `best.pt` / `last.pt`: model, optimizer, configuration, normalization, and training state.
- `history.json`: epoch losses and reconstruction metrics.
- `test_metrics.json` / `test_per_sample.csv`: final test scores and a depth breakdown.

The model returns logits. Apply sigmoid for inclusion probabilities; with the default materials, a continuous conductivity estimate is `0.20 + 0.30 * probability`. These are deterministic voxel predictions, not samples of alternative interiors.
